In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))
import config

try:
    config.assert_data_exists()
    print("✓ Data path:", config.DATA_ROOT)
except FileNotFoundError as e:
    print("✗ Data path error:", e)

import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc, roc_auc_score,
    precision_recall_curve, average_precision_score,
    accuracy_score, f1_score, classification_report,
)
from xgboost import XGBClassifier

sns.set_style("whitegrid")

# Load preprocessed data
SPLIT_PATH = config.DATA_PART2_PROCESSED / "part2_train_test_split.pkl"
if not SPLIT_PATH.exists():
    raise FileNotFoundError(f"Run 02_feature_engineering.ipynb first. Missing: {SPLIT_PATH}")

with open(SPLIT_PATH, "rb") as f:
    data = pickle.load(f)

X_train = data["X_train"]
X_test = data["X_test"]
y_train = data["y_train"]
y_test = data["y_test"]
categorical_cols = data["categorical_cols"]
numeric_cols = data["numeric_cols"]
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

# Use a subset for evaluation speed (set to None for full test set)
EVAL_SAMPLE = 50_000

if EVAL_SAMPLE and len(X_test) > EVAL_SAMPLE:
    idx = np.random.RandomState(42).choice(len(X_test), EVAL_SAMPLE, replace=False)
    X_test = X_test.iloc[idx].reset_index(drop=True)
    y_test = y_test.iloc[idx].reset_index(drop=True)
    print(f"Using {EVAL_SAMPLE:,} test sample")

if EVAL_SAMPLE and len(X_train) > EVAL_SAMPLE:
    idx = np.random.RandomState(42).choice(len(X_train), EVAL_SAMPLE, replace=False)
    X_train = X_train.iloc[idx].reset_index(drop=True)
    y_train = y_train.iloc[idx].reset_index(drop=True)
    print(f"Using {EVAL_SAMPLE:,} train sample")

# 05 — Final Evaluation (Part 2: BTS 2023)
**CMPE 188 | Flight Delay Prediction**

Comprehensive evaluation of trained models:
1. Confusion matrices (XGBoost + Random Forest)
2. ROC curves overlaid
3. Precision-Recall curves
4. Feature importance (XGBoost gain + permutation importance)
5. Final summary table comparing Part 1 vs Part 2 results

Models are re-trained on the sample here for evaluation. For final results,
load the best tuned models from `04_model_tuning.ipynb`.

## 1. Train Models on Sample

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
        ("num", MinMaxScaler(), numeric_cols),
    ]
)
selector = SelectKBest(score_func=chi2, k=80)

# Train XGBoost
xgb = Pipeline([
    ("preprocessor", preprocessor),
    ("selector", selector),
    ("classifier", XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1,
                                  eval_metric="logloss", random_state=42)),
])
print("Training XGBoost...")
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)
xgb_proba = xgb.predict_proba(X_test)[:, 1]
print(f"XGBoost trained. AUC: {roc_auc_score(y_test, xgb_proba):.4f}")

# Train Random Forest
rf = Pipeline([
    ("preprocessor", preprocessor),
    ("selector", selector),
    ("classifier", RandomForestClassifier(n_estimators=100, max_depth=10,
                                           random_state=42, n_jobs=-1)),
])
print("Training Random Forest...")
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:, 1]
print(f"RF trained. AUC: {roc_auc_score(y_test, rf_proba):.4f}")

## 2. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (pred, name) in zip(axes, [(xgb_pred, "XGBoost"), (rf_pred, "Random Forest")]):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["No Delay", "Delayed"],
                yticklabels=["No Delay", "Delayed"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(f"{name} Confusion Matrix")

plt.suptitle("Part 2: Confusion Matrices")
plt.tight_layout()
plt.show()

## 3. ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

models = [
    (xgb_proba, "XGBoost", "steelblue"),
    (rf_proba, "Random Forest", "seagreen"),
]

for proba, name, color in models:
    fpr, tpr, _ = roc_curve(y_test, proba)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc:.4f})", color=color, linewidth=2)

ax.plot([0, 1], [0, 1], "k--", alpha=0.4, label="Random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — Part 2 (BTS 2023)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 4. Precision-Recall Curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

for proba, name, color in models:
    precision, recall, _ = precision_recall_curve(y_test, proba)
    ap = average_precision_score(y_test, proba)
    ax.plot(recall, precision, label=f"{name} (AP = {ap:.4f})", color=color, linewidth=2)

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curves — Part 2")
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

## 5. Feature Importance

In [ ]:
# XGBoost feature importance (gain)
xgb_model = xgb.named_steps["classifier"]

# Get feature names after preprocessing
preprocessor_fitted = xgb.named_steps["preprocessor"]
cat_feature_names = (preprocessor_fitted.named_transformers_["cat"]
                     .get_feature_names_out(categorical_cols))
all_feature_names = np.concatenate([cat_feature_names, numeric_cols])

# After SelectKBest, we only have k features. Get the selected mask.
selector_fitted = xgb.named_steps["selector"]
selected_mask = selector_fitted.get_support()
selected_features = all_feature_names[selected_mask]

# Get feature importances
importances = xgb_model.feature_importances_

# If lengths match, use selected_features; otherwise use generic indices
if len(importances) == len(selected_features):
    feat_names = list(selected_features)
else:
    feat_names = [f"feature_{i}" for i in range(len(importances))]

# Sort and take top 20
sorted_idx = np.argsort(importances)[-20:]

fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(range(len(sorted_idx)), importances[sorted_idx], color="steelblue", edgecolor="white")
ax.set_yticks(range(len(sorted_idx)))
ax.set_yticklabels([feat_names[i] for i in sorted_idx])
ax.set_xlabel("Feature Importance (gain)")
ax.set_title("XGBoost Feature Importance — Top 20 (Part 2)")
plt.tight_layout()
plt.show()

In [ ]:
# Permutation importance (model-agnostic, on test set)
print("Computing permutation importance (this may take a few minutes)...")

# Use a smaller sample for permutation importance
perm_sample = min(5_000, len(X_test))
X_perm = X_test.head(perm_sample)
y_perm = y_test.head(perm_sample)

perm_result = permutation_importance(
    xgb, X_perm, y_perm, n_repeats=5, random_state=42, n_jobs=-1, scoring="roc_auc"
)

perm_importances = perm_result.importances_mean
top20_idx = np.argsort(perm_importances)[-20:]

fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(range(len(top20_idx)), perm_importances[top20_idx],
        color="coral", edgecolor="white",
        xerr=perm_result.importances_std[top20_idx])
ax.set_yticks(range(len(top20_idx)))
ax.set_yticklabels([X_perm.columns[i] for i in top20_idx])
ax.set_xlabel("Permutation Importance (ROC-AUC drop)")
ax.set_title("Permutation Importance — Top 20 Features (Part 2)")
plt.tight_layout()
plt.show()

## 6. Final Summary Table

Comparing Part 1 and Part 2 model performance.

In [ ]:
# Part 1 baseline metrics (from part1 notebooks)
# These are the previously achieved results for reference
part1_results = {
    "XGBoost (Part 1)": {"Accuracy": 0.6438, "ROC-AUC": 0.6895, "F1": 0.51},
    "RF (Part 1)":     {"Accuracy": 0.6384, "ROC-AUC": 0.6848, "F1": 0.44},
}

# Part 2 metrics (from this notebook)
part2_results = {
    "XGBoost (Part 2)": {"Accuracy": accuracy_score(y_test, xgb_pred), "ROC-AUC": roc_auc_score(y_test, xgb_proba), "F1": f1_score(y_test, xgb_pred)},
    "RF (Part 2)":     {"Accuracy": accuracy_score(y_test, rf_pred), "ROC-AUC": roc_auc_score(y_test, rf_proba), "F1": f1_score(y_test, rf_pred)},
    "MLP (Part 2)":    {"Accuracy": 0.0, "ROC-AUC": 0.0, "F1": 0.0},  # Fill from notebook 06
}

all_results = {**part1_results, **part2_results}
summary = pd.DataFrame(all_results).T
summary.index.name = "Model"
print("Model Performance Comparison")
print("=" * 60)
print(summary.round(4).to_string())
print()
print("Note: Part 1 uses Airlines.csv (2008-2011, 540K rows, climate averages).")
print("Note: Part 2 uses BTS 2023 (6.7M rows, daily weather, aircraft info).")
print("MLP results to be filled from 06_neural_network.ipynb.")